In [14]:
import os

In [15]:
def search_wavs(rootdir):
    paths=[]
    for root, dirs, files in os.walk(rootdir):
        for file in files:
            if file.endswith(".wav"):
                paths.append(os.path.join(root,file))
    
    return paths

In [4]:
oracles=search_wavs('oracle')

In [16]:
import numpy as np

In [9]:
np.random.choice(oracles,50,replace=False)

array(['oracle/p261/p261_001.wav', 'oracle/p362/p362_005.wav',
       'oracle/p238/p238_004.wav', 'oracle/p294/p294_001.wav',
       'oracle/p252/p252_005.wav', 'oracle/p238/p238_001.wav',
       'oracle/p343/p343_004.wav', 'oracle/p252/p252_002.wav',
       'oracle/p360/p360_001.wav', 'oracle/p238/p238_002.wav',
       'oracle/p238/p238_003.wav', 'oracle/p243/p243_005.wav',
       'oracle/p241/p241_004.wav', 'oracle/p362/p362_002.wav',
       'oracle/p360/p360_004.wav', 'oracle/p334/p334_004.wav',
       'oracle/p362/p362_003.wav', 'oracle/p334/p334_002.wav',
       'oracle/p360/p360_002.wav', 'oracle/p360/p360_005.wav',
       'oracle/p334/p334_003.wav', 'oracle/p261/p261_003.wav',
       'oracle/p243/p243_001.wav', 'oracle/p243/p243_002.wav',
       'oracle/p241/p241_002.wav', 'oracle/p294/p294_002.wav',
       'oracle/p252/p252_004.wav', 'oracle/p294/p294_004.wav',
       'oracle/p243/p243_003.wav', 'oracle/p261/p261_005.wav',
       'oracle/p343/p343_002.wav', 'oracle/p343/p343_00

In [12]:
a=[]
a.extend(oracles)
a

['oracle/p334/p334_003.wav',
 'oracle/p334/p334_002.wav',
 'oracle/p334/p334_001.wav',
 'oracle/p334/p334_004.wav',
 'oracle/p334/p334_005.wav',
 'oracle/p238/p238_003.wav',
 'oracle/p238/p238_005.wav',
 'oracle/p238/p238_001.wav',
 'oracle/p238/p238_002.wav',
 'oracle/p238/p238_004.wav',
 'oracle/p360/p360_005.wav',
 'oracle/p360/p360_002.wav',
 'oracle/p360/p360_001.wav',
 'oracle/p360/p360_004.wav',
 'oracle/p360/p360_003.wav',
 'oracle/p243/p243_002.wav',
 'oracle/p243/p243_005.wav',
 'oracle/p243/p243_001.wav',
 'oracle/p243/p243_003.wav',
 'oracle/p243/p243_004.wav',
 'oracle/p241/p241_004.wav',
 'oracle/p241/p241_001.wav',
 'oracle/p241/p241_003.wav',
 'oracle/p241/p241_002.wav',
 'oracle/p241/p241_005.wav',
 'oracle/p343/p343_004.wav',
 'oracle/p343/p343_003.wav',
 'oracle/p343/p343_001.wav',
 'oracle/p343/p343_005.wav',
 'oracle/p343/p343_002.wav',
 'oracle/p261/p261_001.wav',
 'oracle/p261/p261_005.wav',
 'oracle/p261/p261_002.wav',
 'oracle/p261/p261_003.wav',
 'oracle/p261/

In [17]:
import shutil

In [14]:
shutil.copy(a[0],'s3bucket/1.wav')


's3bucket/1.wav'

In [18]:
dirmap={"oracle":"oracle",
        "diffvc":"outputs_diffvc",
        "diffvc_p":"outputs_diffvcplus",
        "diffvc_pl":"outputs_diffvcp_light",
        "diffvc_pd":"outputs_diffvcp_ctrl"}

In [17]:
list(dirmap.keys())

['oracle', 'diffvc', 'diffvc_p', 'diffvc_pl', 'diffvc_pd']

In [20]:
os.makedirs('s3bucket',exist_ok=True)
for method in list(dirmap.keys()):
    os.makedirs(f's3bucket/{method}',exist_ok=True)
    wavs=search_wavs(dirmap[method])
    subset=np.random.choice(wavs, 30, replace=False)
    for i,wav in enumerate(subset):
        shutil.copy(wav,f's3bucket/{method}') #{method}_{i}.wav

In [36]:
import librosa
from scipy.io import wavfile
def normalize_wav(wav_path):
    audio, sr=librosa.load(wav_path)
    audio = librosa.util.normalize(audio) * 0.95
    wavfile.write(wav_path, sr, (audio * 32767).astype("int16"))

In [37]:
for wav_path in search_wavs('s3bucket'):
    normalize_wav(wav_path)

In [19]:
males=['p241','p243','p252','p334','p360']
females=['p238','p261','p294','p343','p362']
utts=['001','002','003','004','005']

In [46]:
ms=males.copy()
ms[0]='0'
ms,males,len(ms)

(['0', 'p243', 'p252', 'p334', 'p360'],
 ['p241', 'p243', 'p252', 'p334', 'p360'],
 5)

In [20]:
def gen_spkmap(males=['p241','p243','p252','p334','p360'],
               females=['p238','p261','p294','p343','p362']):
    spkmap={}
    ms=males.copy()
    fs=females.copy()
    for method in list(dirmap.keys()):
        # print(ms,fs)
        if len(ms)<2:
            ms=males.copy()
            fs=females.copy()  
            # print('reset')
            # print(ms,fs)
        m2=list(np.random.choice(ms,2,replace=False))
        f2=list(np.random.choice(fs,2,replace=False))
        ms=[m for m in ms if m not in m2]
        fs=[f for f in fs if f not in f2]
        # print(m2,f2)
        spkmap[method]=(m2,f2)
    return spkmap

In [21]:
gen_spkmap()

{'oracle': (['p334', 'p252'], ['p238', 'p294']),
 'diffvc': (['p241', 'p360'], ['p362', 'p343']),
 'diffvc_p': (['p334', 'p252'], ['p238', 'p261']),
 'diffvc_pl': (['p360', 'p241'], ['p343', 'p362']),
 'diffvc_pd': (['p334', 'p241'], ['p294', 'p261'])}

In [23]:
m2=['p252', 'p243']
f2=['p362', 'p294']
m2,f2

(['p252', 'p243'], ['p362', 'p294'])

In [53]:
# m2=['p252', 'p243']
# f2=['p362', 'p294']
# def gen_pairs(m2,f2,utts=['001','002','003','004','005']):
#     print(m2,f2)
#     us=utts.copy()
#     for m in m2:
#         for f in f2:
#             print(us)
#             if len(us)<2:
#                 us=utts.copy()
#                 print('reset')
#                 print(us)
#             u2=np.random.choice(us,2,replace=False)
#             us=[u for u in us if u not in u2]
#             print(u2)
#             print(f'{m}_{f}_{u2[0]}',f'{f}_{m}_{u2[1]}')
#     if len(us)<2:
#         us=utts.copy()
#         print('reset')
#         print(us)
#     u2=np.random.choice(us,2,replace=False)
#     us=[u for u in us if u not in u2]
#     print(u2)
#     print(f'{m2[0]}_{m2[1]}_{u2[0]}',f'{m2[1]}_{m2[0]}_{u2[1]}')
#     if len(us)<2:
#         us=utts.copy()
#         print('reset')
#         print(us)
#     u2=np.random.choice(us,2,replace=False)
#     us=[u for u in us if u not in u2]
#     print(u2)
#     print(f'{f2[0]}_{f2[1]}_{u2[0]}',f'{f2[1]}_{f2[0]}_{u2[1]}')

# gen_pairs(m2,f2)

['p252' 'p243'] ['p362' 'p294']
['001', '002', '003', '004', '005']
['004' '002']
p252_p362_004 p362_p252_002
['001', '003', '005']
['003' '005']
p252_p294_003 p294_p252_005
['001']
reset
['001', '002', '003', '004', '005']
['001' '003']
p243_p362_001 p362_p243_003
['002', '004', '005']
['004' '005']
p243_p294_004 p294_p243_005
reset
['001', '002', '003', '004', '005']
['004' '001']
p252_p243_004 p243_p252_001
['005' '003']
p362_p294_005 p294_p362_003


In [24]:
def gen_vitalpairs(m2,f2,utts=['001','002','003','004','005']):
    pairs=[]
    us=utts.copy()
    for m in m2:
        for f in f2:
            if len(us)<2:
                us=utts.copy()
            u2=np.random.choice(us,2,replace=False)
            us=[u for u in us if u not in u2]
            pairs.append(f'{m}_{f}_{u2[0]}.wav')
            pairs.append(f'{f}_{m}_{u2[1]}.wav')
    if len(us)<2:
        us=utts.copy()
    u2=np.random.choice(us,2,replace=False)
    us=[u for u in us if u not in u2]
    pairs.append(f'{m2[0]}_{m2[1]}_{u2[0]}.wav')
    pairs.append(f'{m2[1]}_{m2[0]}_{u2[1]}.wav')
    if len(us)<2:
        us=utts.copy()
    u2=np.random.choice(us,2,replace=False)
    us=[u for u in us if u not in u2]
    pairs.append(f'{f2[0]}_{f2[1]}_{u2[0]}.wav')
    pairs.append(f'{f2[1]}_{f2[0]}_{u2[1]}.wav')
    return pairs
gen_vitalpairs(m2,f2),len(gen_vitalpairs(m2,f2))

(['p252_p362_004.wav',
  'p362_p252_002.wav',
  'p252_p294_005.wav',
  'p294_p252_001.wav',
  'p243_p362_005.wav',
  'p362_p243_003.wav',
  'p243_p294_001.wav',
  'p294_p243_004.wav',
  'p252_p243_001.wav',
  'p243_p252_005.wav',
  'p362_p294_004.wav',
  'p294_p362_002.wav'],
 12)

In [25]:
gen_spkmap()

{'oracle': (['p252', 'p360'], ['p362', 'p261']),
 'diffvc': (['p243', 'p334'], ['p238', 'p294']),
 'diffvc_p': (['p241', 'p334'], ['p343', 'p261']),
 'diffvc_pl': (['p243', 'p360'], ['p362', 'p294']),
 'diffvc_pd': (['p243', 'p360'], ['p238', 'p362'])}

In [26]:
def helper(method,utt):
    spk=utt.split('_')[0]
    return f'{dirmap[method]}/{spk}/{utt}'

helper('oracle','p294_002.wav')

'oracle/p294/p294_002.wav'

In [35]:
######################################
os.makedirs('s3bucket',exist_ok=True)
spkmap=gen_spkmap()
for method in list(dirmap.keys()):

    os.makedirs(f's3bucket/{method}',exist_ok=True)

    if method=='oracle':
        wavs=search_wavs(dirmap[method])
        subset=np.random.choice(wavs, 20, replace=False)
        for i,wav in enumerate(subset):
            shutil.copy(wav,f's3bucket/{method}') 
        continue
    
    m2,f2=spkmap[method]
    print(method,m2,f2)
    subset_=gen_vitalpairs(m2,f2)
    subset=[helper(method,i) for i in subset_]
    print(len(subset), subset)
    wavs=search_wavs(dirmap[method])
    while len(subset)<20:
        awav=np.random.choice(wavs, 1)[0]
        if awav not in subset:
            flag = True
            for m in m2:
                for f in f2:
                    if (f in awav) or (m in awav):
                        flag = False
            if flag:
                # print(awav)
                subset.append(awav)    

    print(subset)
    
    for i,wav in enumerate(subset):
        shutil.copy(wav,f's3bucket/{method}') #{method}_{i}.wav
    # break
    

diffvc ['p252', 'p360'] ['p362', 'p261']
12 ['outputs_diffvc/p252/p252_p362_004.wav', 'outputs_diffvc/p362/p362_p252_005.wav', 'outputs_diffvc/p252/p252_p261_002.wav', 'outputs_diffvc/p261/p261_p252_003.wav', 'outputs_diffvc/p360/p360_p362_001.wav', 'outputs_diffvc/p362/p362_p360_004.wav', 'outputs_diffvc/p360/p360_p261_002.wav', 'outputs_diffvc/p261/p261_p360_003.wav', 'outputs_diffvc/p252/p252_p360_001.wav', 'outputs_diffvc/p360/p360_p252_002.wav', 'outputs_diffvc/p362/p362_p261_004.wav', 'outputs_diffvc/p261/p261_p362_003.wav']
['outputs_diffvc/p252/p252_p362_004.wav', 'outputs_diffvc/p362/p362_p252_005.wav', 'outputs_diffvc/p252/p252_p261_002.wav', 'outputs_diffvc/p261/p261_p252_003.wav', 'outputs_diffvc/p360/p360_p362_001.wav', 'outputs_diffvc/p362/p362_p360_004.wav', 'outputs_diffvc/p360/p360_p261_002.wav', 'outputs_diffvc/p261/p261_p360_003.wav', 'outputs_diffvc/p252/p252_p360_001.wav', 'outputs_diffvc/p360/p360_p252_002.wav', 'outputs_diffvc/p362/p362_p261_004.wav', 'outputs_di

In [31]:
'p294' in 'outputs_diffvc/p294/p294_p241_001.wav'

True

In [3]:
import numpy as np
np.random.choice([1,2,3,4],1) in [1,2,3,4]

True

In [12]:
np.array([5])[0]

5

In [11]:
'p251' in 'p252_p243_001.wav'

False

In [82]:
for dir in os.listdir('s3bucket'):
    print(len(search_wavs(f's3bucket/{dir}')))

30
30
30
30
30


In [87]:
shuffled=search_wavs('s3bucket')
np.random.shuffle(shuffled)
shuffled

['s3bucket/diffvc_pl/p243_p334_003.wav',
 's3bucket/oracle/p241_004.wav',
 's3bucket/diffvc_pl/p334_p343_005.wav',
 's3bucket/diffvc_pl/p238_p241_003.wav',
 's3bucket/diffvc_p/p343_p243_003.wav',
 's3bucket/diffvc_pl/p243_p238_002.wav',
 's3bucket/diffvc_pd/p362_p261_005.wav',
 's3bucket/diffvc/p238_p243_005.wav',
 's3bucket/diffvc_p/p360_p241_001.wav',
 's3bucket/diffvc_p/p360_p261_004.wav',
 's3bucket/diffvc_pl/p238_p261_002.wav',
 's3bucket/diffvc_pl/p241_p360_005.wav',
 's3bucket/diffvc_pd/p252_p238_004.wav',
 's3bucket/diffvc/p238_p241_003.wav',
 's3bucket/diffvc_p/p294_p334_003.wav',
 's3bucket/oracle/p261_002.wav',
 's3bucket/diffvc/p261_p343_005.wav',
 's3bucket/diffvc_pd/p334_p261_005.wav',
 's3bucket/diffvc_pd/p252_p343_005.wav',
 's3bucket/oracle/p343_002.wav',
 's3bucket/diffvc_pd/p343_p243_002.wav',
 's3bucket/diffvc_pl/p238_p360_001.wav',
 's3bucket/diffvc_pd/p238_p241_005.wav',
 's3bucket/diffvc/p241_p360_003.wav',
 's3bucket/diffvc_p/p334_p243_002.wav',
 's3bucket/diffv

In [88]:
import csv

header = ['audio_url']

with open('input.csv', 'w', encoding='utf-8') as file_obj:
    writer = csv.writer(file_obj)
    writer.writerow(header)
    for wav_path in shuffled:
        writer.writerow((wav_path,))